# Lab Exercise: Multi-Agent Reinforcement Learning with BenchMARL and VMAS

BenchMARL is a research framework for running and comparing multi-agent reinforcement learning experiments with a consistent workflow. It connects algorithms, environments, models, logging, and evaluation into one configurable pipeline, which makes it useful for both research experiments and classroom demonstrations.

In this lab, you will use BenchMARL with VMAS environments to train agents, save experiment results, and inspect the learning curves. The goal is to have a simple, reliable notebook workflow that can be extended later with deeper algorithm explanations and more structured experiments.

VMAS, short for Vectorized Multi-Agent Simulator, is the simulator that provides the multi-agent tasks used in this lab. BenchMARL treats VMAS tasks through the same interface it uses for other environments, which makes it easier to swap tasks or compare different training setups.

## Setup

The next cell follows the official BenchMARL Colab setup: it clones the original BenchMARL repository, moves into it, installs and updates everything needed, and installs VMAS so the VMAS tasks are available in the notebook.

In [ ]:
import os

if not os.path.isdir("/content/BenchMARL"):
    !git clone https://github.com/facebookresearch/BenchMARL /content/BenchMARL

In [ ]:
%cd /content/BenchMARL

In [ ]:
!pip install -U torch torchvision

In [ ]:
!pip install -e .

In [ ]:
!pip install vmas

In [ ]:
!apt-get update
!apt-get install -y x11-utils python3-opengl xvfb
!pip install pyvirtualdisplay
import pyvirtualdisplay

display = pyvirtualdisplay.Display(visible=False, size=(1400, 900))
display.start()

In [ ]:
from pathlib import Path
import time
import json
import pandas as pd
import matplotlib.pyplot as plt
import torch

## BenchMARL imports

The imports below bring in the main BenchMARL building blocks used in this notebook:


- `benchmarl.environments`: environment/task definitions such as `VmasTask`; these select the VMAS scenario the agents will train in.
- `benchmarl.algorithms`: algorithm configurations such as `MappoConfig` and `MasacConfig`; these define the training strategy.
- `benchmarl.models`: model configurations such as `MlpConfig`; these define the neural networks used by the agents, for example policies and critics.
- `benchmarl.experiment`: `Experiment` and `ExperimentConfig`; these combine the task, algorithm, model, seed, logging, evaluation, and runtime settings into one training run.


For more detail about these BenchMARL components, see the local [README.md](README.md).

In [ ]:
from benchmarl.environments import VmasTask
from benchmarl.algorithms import MappoConfig, MasacConfig, QmixConfig, VdnConfig, MaddpgConfig
from benchmarl.models import MlpConfig, CnnConfig, GnnConfig
from benchmarl.experiment import Experiment, ExperimentConfig

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


def algorithm_name(algorithm_config):
    return algorithm_config.__name__.replace("Config", "").upper()

## Core Parameters

The next cell contains the main values you can safely change for a quick experiment. The defaults are intentionally small, so the notebook can run as a test in Colab without taking too long.

- `SELECTED_TASK`: the VMAS task to train on, chosen directly from the `VmasTask` enum.
- `ALGORITHM_CONFIGS`: the BenchMARL algorithm configuration classes to compare.
- `MODEL_CONFIG`: the neural network architecture used by the agents.
- `SEED`: controls randomness, so repeated runs can be compared more fairly.
- `MAX_ITERS`: how many training iterations to run; increase this for better learning curves.
- `FRAMES_PER_BATCH`: how many environment frames are collected per iteration. In BenchMARL, a frame is one environment interaction step collected during training.
- `N_ENVS`: how many VMAS environments run in parallel during data collection.
- `OFFPOLICY_OPT_STEPS`: optimizer steps used by off-policy algorithms such as MASAC.
- `MINIBATCH_SIZE`: how many collected frames are used in one training minibatch. Larger values can make updates more stable, but use more memory.
- `RENDER`: whether to render evaluation episodes. Keep this `False` for faster training, and turn it on only when you want to visually inspect the learned behavior.
- `EVAL_FREQUENCY`: how often evaluation runs, measured in training iterations. In the code below it is converted to BenchMARL frames as `EVAL_FREQUENCY * FRAMES_PER_BATCH`.
- `EVAL_EPISODES`: how many episodes are used for each evaluation point.
- `TASK_MAX_STEPS`: maximum length of one VMAS episode. Tasks where agents need more time to reach the objective should use a higher value, otherwise the episode may end before the agents have enough steps to complete the task.

These are only the parameters exposed for this lab. To see the full list of experiment parameters that BenchMARL can control, open [base_experiment.yaml](../../benchmarl/conf/experiment/base_experiment.yaml), where each parameter is documented with a short explanation.

In [ ]:
SELECTED_TASK = VmasTask.BALANCE
ALGORITHM_CONFIGS = [MappoConfig, MasacConfig]  # For a faster run, use [MappoConfig]
MODEL_CONFIG = MlpConfig
SEED = 0

# Classroom-friendly runtime
MAX_ITERS = 50
FRAMES_PER_BATCH = 6000
N_ENVS = 10
OFFPOLICY_OPT_STEPS = 20
MINIBATCH_SIZE = 400

RENDER = False
EVAL_FREQUENCY = 10  # (evaluation every 10 training iterations)
EVAL_EPISODES = 10

TASK_MAX_STEPS = 150

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_ROOT = Path("/content/student_lab_runs")
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

assert hasattr(SELECTED_TASK, "get_from_yaml")
for algorithm_config in ALGORITHM_CONFIGS:
    assert hasattr(algorithm_config, "get_from_yaml")

print("TASK:", SELECTED_TASK.name)
print("ALGORITHMS:", [algorithm_name(algorithm_config) for algorithm_config in ALGORITHM_CONFIGS])
print("DEVICE:", DEVICE)
print("SAVE_ROOT:", SAVE_ROOT)

In [ ]:
def make_experiment(algorithm_config) -> Experiment:
    """Create one BenchMARL experiment from the lab settings.

    The function loads the default BenchMARL experiment, task, algorithm,
    and model configurations, then overrides the parts users control in
    the parameter cell above. The returned `Experiment` is ready to run with
    `exp.run()`.
    """
    exp_cfg = ExperimentConfig.get_from_yaml()

    # Device
    exp_cfg.train_device = DEVICE

    exp_cfg.max_n_iters = MAX_ITERS
    exp_cfg.max_n_frames = None

    # On Policy settings (used by for example MAPPO)
    exp_cfg.on_policy_collected_frames_per_batch = FRAMES_PER_BATCH
    exp_cfg.on_policy_n_envs_per_worker = N_ENVS
    exp_cfg.on_policy_minibatch_size = MINIBATCH_SIZE

    # Off Policy settings (used by for example MASAC)
    exp_cfg.off_policy_collected_frames_per_batch = FRAMES_PER_BATCH
    exp_cfg.off_policy_n_envs_per_worker = N_ENVS
    exp_cfg.off_policy_n_optimizer_steps = OFFPOLICY_OPT_STEPS

    # Logging
    exp_cfg.loggers = ["csv"]
    exp_cfg.create_json = True
    exp_cfg.save_folder = str(SAVE_ROOT)

    # Evaluation
    exp_cfg.evaluation = True
    exp_cfg.evaluation_episodes = EVAL_EPISODES
    exp_cfg.evaluation_interval = EVAL_FREQUENCY * FRAMES_PER_BATCH
    exp_cfg.render = RENDER

    task = SELECTED_TASK.get_from_yaml()
    task.config["max_steps"] = TASK_MAX_STEPS

    algo_cfg = algorithm_config.get_from_yaml()
    model_cfg = MODEL_CONFIG.get_from_yaml()
    critic_cfg = MODEL_CONFIG.get_from_yaml()

    experiment = Experiment(
        task=task,
        algorithm_config=algo_cfg,
        model_config=model_cfg,
        seed=SEED,
        config=exp_cfg,
        critic_model_config=critic_cfg,
    )
    return experiment

## Run the selected experiments

This cell creates and runs one BenchMARL experiment for each algorithm selected in `ALGORITHM_CONFIGS`. It also records the algorithm name, output folder, run name, and runtime so we can inspect and compare the results later.

In [ ]:
runs = []

for algorithm_config in ALGORITHM_CONFIGS:
    label = algorithm_name(algorithm_config)
    print(f"\n=== Start: {label} ===")
    exp = make_experiment(algorithm_config)

    t0 = time.time()
    exp.run()
    dt = time.time() - t0

    runs.append({
        "algorithm": label,
        "run_name": exp.name,
        "run_folder": str(exp.folder_name),
        "seconds": round(dt, 1),
    })

pd.DataFrame(runs)

## Reading the reward plot

The next cell reads the CSV scalar logs saved by BenchMARL for each run, loads `eval_reward_episode_reward_mean.csv`, and plots the evaluation reward curve for every selected algorithm. This gives a quick visual comparison of how the agents performed during training.

The **X axis** is the training step/frame count at which evaluation was logged. The **Y axis** is the mean episode reward during evaluation. In general, a curve that moves upward means the agents are learning behavior that receives better reward. A flat curve, a very noisy curve, or a curve that improves and then collapses usually means the setup needs more investigation.

If you do not see improvement, first try changing experiment parameters such as `MAX_ITERS`, `FRAMES_PER_BATCH`, `MINIBATCH_SIZE`, `N_ENVS`, `EVAL_FREQUENCY`, or `TASK_MAX_STEPS`. For example, increasing `MAX_ITERS` gives the agents more training time, while increasing `TASK_MAX_STEPS` gives them more steps inside each episode to finish longer tasks.

**Important:** sometimes the problem is not only the training parameters. The task itself and its reward design can limit how well the agents can solve the objective. In those cases, a full solution may require changing values inside `task.config`, not only the experiment settings. For example, inside `make_experiment` you can adjust task-specific configuration after loading the task:

```python
task = SELECTED_TASK.get_from_yaml()
task.config["max_steps"] = TASK_MAX_STEPS
task.config["n_agents"] = 3          # example: change a task-specific parameter
task.config["shared_reward"] = True  # example: change reward/task behavior if supported
```

The available keys depend on the selected VMAS task, so inspect the task YAML/config before changing them. Start with small changes and compare the reward plot again.

In [ ]:
plt.figure(figsize=(8, 5))

for r in runs:
    run_folder = Path(r["run_folder"])
    run_name = r["run_name"]
    csv_path = run_folder / run_name / "scalars" / "eval_reward_episode_reward_mean.csv"

    if not csv_path.exists():
        print("Missing CSV:", csv_path)
        continue

    df = pd.read_csv(csv_path, header=None, names=["step", "reward"])
    plt.plot(df["step"], df["reward"], marker="o", label=r["algorithm"])

plt.title(f"Reward comparison (task={SELECTED_TASK.name})")
plt.xlabel("Step")
plt.ylabel("Eval episode reward mean")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## Example exercises

Try one or more of the following mini-experiments. For each one, write down what you changed, rerun the notebook cells from the parameter cell onward, and compare the reward plot.


1. **Change evaluation behavior.** Set `EVAL_FREQUENCY` to a smaller value so evaluation happens more often. Then set it to a larger value and compare how much detail you lose or gain in the plot.
2. **Inspect learned behavior visually.** After a short training run, set `RENDER = True` for evaluation. Observe whether the visual behavior matches what the reward curve suggests.
3. **Explore task configuration.** Pick one task-specific value from `task.config`, change it inside `make_experiment`, and explain how it changes the difficulty or objective of the task. Start with a small change, then rerun and compare the plot.

## Conclusion

In this lab, you set up BenchMARL in Colab, selected a VMAS task, created BenchMARL experiments from reusable configuration objects, and ran one or more MARL algorithms through the same workflow. You also inspected where BenchMARL saves its logs and used the saved CSV scalar files to plot evaluation reward over training.

The most important idea is that BenchMARL separates the main pieces of a MARL experiment: the task, the algorithm, the model, and the experiment configuration. This makes it easier to change one part at a time and compare the results fairly.

When the reward curve does not improve, do not assume the algorithm simply failed. First check the experiment parameters, then think about whether the task configuration and reward design give the agents enough time, information, and incentive to complete the objective. Good MARL experiments usually come from careful iteration, not from a single perfect run.